# DogID - Eksploracyjna analiza danych

Notebook służy do zrozumienia struktury zbioru Stanford Dogs (15 ras) przed treningiem oraz oceny modelu po treningu. Każda sekcja generuje konkretny artefakt wizualny przydatny w prezentacji projektu.

**Wymagania:** 
- pobrane dane (`python -m data.download`)
- (sekcja 4) wytrenowany model (`python -m model.train`)

**Uruchomienie:**
```bash
jupyter notebook notebooks/data_exploration.ipynb
```

In [ ]:
import random
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from PIL import Image

# Notebook żyje w notebooks/, dodajemy root projektu do PYTHONPATH
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data.breeds import BREEDS, num_classes  # noqa: E402
from data.preprocess import train_transforms  # noqa: E402
from model.architecture import load_trained_model  # noqa: E402

DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
WEIGHTS_PATH = PROJECT_ROOT / 'models' / 'dogid.pt'

sns.set_theme(style='whitegrid')
random.seed(42)
print(f'Wspieranych ras: {num_classes()}')
print(f'Dane: {DATA_DIR}')
print(f'Model: {WEIGHTS_PATH} (istnieje: {WEIGHTS_PATH.exists()})')

## 1. Rozkład klas

Sprawdzamy, czy zbiór jest zbalansowany. Wyraźna dysproporcja byłaby sygnałem, że model może mieć tendencję do faworyzowania klas większych.

In [ ]:
def count_images_per_breed(split: str) -> dict[str, int]:
    """Zlicza obrazy per rasa w podanym splicie ('train'/'val'/'test')."""
    counts = {}
    for breed in BREEDS:
        breed_dir = DATA_DIR / split / breed.folder_name
        counts[breed.name_pl] = len(list(breed_dir.glob('*.jpg')))
    return counts


splits = ['train', 'val', 'test']
data = {split: count_images_per_breed(split) for split in splits}
breed_names = list(data['train'].keys())

fig, ax = plt.subplots(figsize=(12, 6))
x_positions = np.arange(len(breed_names))
bar_width = 0.27

for offset, split in enumerate(splits):
    values = list(data[split].values())
    ax.bar(x_positions + offset * bar_width, values, bar_width, label=split)

ax.set_xticks(x_positions + bar_width)
ax.set_xticklabels(breed_names, rotation=45, ha='right')
ax.set_ylabel('Liczba obrazów')
ax.set_title('Rozkład klas w zbiorach train / val / test')
ax.legend()
plt.tight_layout()
plt.show()

print('\nSumarycznie per split:')
for split in splits:
    total = sum(data[split].values())
    print(f'  {split}: {total}')

## 2. Przykładowe obrazy

Po jednym losowym zdjęciu z każdej rasy w siatce 5 x 3 - szybkie spojrzenie na wizualną różnorodność klas.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for ax, breed in zip(axes.flat, BREEDS):
    breed_dir = DATA_DIR / 'train' / breed.folder_name
    sample_path = random.choice(list(breed_dir.glob('*.jpg')))
    image = Image.open(sample_path)
    ax.imshow(image)
    ax.set_title(breed.name_pl, fontsize=10)
    ax.axis('off')
plt.suptitle('Po jednym zdjęciu z każdej rasy (losowy wybór)', y=1.01)
plt.tight_layout()
plt.show()

## 3. Augmentacja - przed i po

Wizualne sprawdzenie, jak transformacje (random crop, flip, rotacja, color jitter) wpływają na obraz. Augmentacja sprzyja generalizacji modelu, ale zbyt agresywna mogłaby psuć rozpoznawalność rasy.

In [ ]:
def denormalize(tensor: torch.Tensor) -> np.ndarray:
    """Odwraca normalizację ImageNet i zamienia tensor na obraz HxWx3 [0,1]."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


transform = train_transforms()
sample_breed = random.choice(BREEDS)
sample_path = random.choice(list((DATA_DIR / 'train' / sample_breed.folder_name).glob('*.jpg')))
original = Image.open(sample_path).convert('RGB')

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes[0, 0].imshow(original)
axes[0, 0].set_title('Oryginał')
axes[0, 0].axis('off')
for ax in axes[0, 1:]:
    ax.axis('off')

for ax in axes[1]:
    augmented = transform(original)
    ax.imshow(denormalize(augmented))
    ax.set_title('Augmentacja')
    ax.axis('off')

plt.suptitle(f'Przykład augmentacji - rasa: {sample_breed.name_pl}', y=1.0)
plt.tight_layout()
plt.show()

## 4. Confusion matrix na zbiorze testowym

Pokazuje, **które rasy model myli ze sobą**. Diagonal = poprawne predykcje, off-diagonal = błędy. To najmocniejszy slajd na obronę projektu - pozwala dyskutować o trudnych przypadkach.

Wymaga wytrenowanego modelu w `models/dogid.pt`.

In [ ]:
from sklearn.metrics import confusion_matrix  # noqa: E402

from data.loader import build_loader  # noqa: E402

model = load_trained_model(str(WEIGHTS_PATH), num_classes=num_classes())
test_loader = build_loader('test')

y_true: list[int] = []
y_pred: list[int] = []
with torch.no_grad():
    for inputs, targets in test_loader:
        logits = model(inputs)
        preds = logits.argmax(dim=1)
        y_true.extend(targets.tolist())
        y_pred.extend(preds.tolist())

matrix = confusion_matrix(y_true, y_pred)
labels = [breed.name_pl for breed in BREEDS]

plt.figure(figsize=(11, 9))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, cbar=False)
plt.title('Confusion matrix - zbiór testowy')
plt.xlabel('Predykcja modelu')
plt.ylabel('Rzeczywista rasa')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

accuracy = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)
print(f'\nTest accuracy: {accuracy * 100:.1f}% ({len(y_true)} obrazów)')